# AI Enterprise Workflow Capstone: Exploratory Data Analysis (EDA)
**Project Title:** AI Enterprise Revenue Forecasting & Production Monitoring Microservice

This notebook conducts a visual exploratory data analysis (EDA) of the transactional invoice stream data,
evaluating revenue distributions, country volume, day-of-week seasonality, and continuous time-series patterns.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_ingestion import fetch_data, convert_to_ts

# Ingest raw transactions and aggregate continuous time-series
df_raw = fetch_data()
ts_all = convert_to_ts(df_raw, country='all')
print(f'Raw transactions count: {len(df_raw)}')
print(f'Daily time-series points: {len(ts_all)}')
df_raw.head()

### 1. Revenue & Transaction Volume by Country
Analyzing key enterprise markets across invoice counts and total monetary volume.

In [2]:
top_country_stats = df_raw.groupby('country').agg(
    total_revenue=('price', 'sum'),
    total_invoices=('invoice', 'count')
).sort_values(by='total_revenue', ascending=False).head(6)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
x = np.arange(len(top_country_stats))
width = 0.35
ax1.bar(x - width/2, top_country_stats['total_revenue'], width, label='Total Revenue ($)', color='#1f77b4')
ax2.bar(x + width/2, top_country_stats['total_invoices'], width, label='Invoices Count', color='#ff7f0e')
ax1.set_xlabel('Country')
ax1.set_ylabel('Revenue ($)', color='#1f77b4')
ax2.set_ylabel('Invoice Count', color='#ff7f0e')
ax1.set_xticks(x)
ax1.set_xticklabels([c.replace('_', ' ').title() for c in top_country_stats.index], rotation=15)
plt.title('Revenue and Transaction Volume by Top 6 Countries')
plt.tight_layout()
plt.show()

### 2. Time-Series Daily Revenue Trends & 30-Day Moving Average

In [3]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts_all['date'], ts_all['revenue'], label='Daily Revenue', alpha=0.45, color='#7f7f7f')
ax.plot(ts_all['date'], ts_all['revenue_roll_mean_30'], label='30-Day Rolling Mean', color='#d62728', linewidth=2.2)
ax.set_title('Daily Revenue Timeline & 30-Day Smoothing Trend (All Countries)')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Revenue ($)')
ax.legend()
plt.tight_layout()
plt.show()